In [1]:
import torch
from sklearn.metrics import r2_score, mean_squared_error

from kids.catalog.preparation import load_and_prepare_training_catalog, merge_training_catalogs
from kids.ml.training_data import TrainingData
from kids.ml.photoz.regressors import BNConfig, BayesianNet
from kids.catalog.features import PHOTOZ_FEATURES

In [17]:
kids_x_desi_path = '../data/KiDS_DR5_WCScut_x_DESI_DR1_columns.csv'
kids_x_sdss_path = '../data/KiDS_DR5_WCScut_x_SDSS_DR17_columns.csv'
model_path = '../saved_models/photoz/bnn_photoz_2026-11-02T16-15-37Z.pt'

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [19]:
model = BayesianNet(BNConfig())
model.load_weights(model_path, device)
model = model.to(device)

In [5]:
kids_x_desi = load_and_prepare_training_catalog(kids_x_desi_path)
kids_x_sdss = load_and_prepare_training_catalog(kids_x_sdss_path)
merged_catalogs = merge_training_catalogs(kids_x_desi, kids_x_sdss)

In [6]:
training_data = TrainingData(merged_catalogs, features=PHOTOZ_FEATURES, train_size=0.8)
X_test, y_test = training_data.get_test_photoz()
X_test_tensor, y_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device), torch.tensor(y_test.values, dtype=torch.float32, device=device)

In [7]:
def redshift_error(y_true, y_pred):
    nominator = y_pred - y_true
    denominator = y_true + 1
    redshift_err = nominator / denominator 
    return redshift_err.mean(), redshift_err.std()

In [20]:
model.eval()
with torch.no_grad():
    photo_z, photo_z_std, nlls = [], [], []
    for _ in range(100):
        preds, _ = model(X_test_tensor)
        photo_z.append(preds.mean.cpu().numpy()), photo_z_std.append(preds.stddev.cpu().numpy())
        nlls.append(-preds.log_prob(y_test_tensor).mean().item())

In [21]:
import numpy as np
y_pred = np.array(photo_z).mean(axis=0)
y_pred_std = np.array(photo_z).std(axis=0)
y_mean_std = np.array(photo_z_std).mean(axis=0)
y_std = np.sqrt(y_pred_std**2 + y_mean_std**2)
nll = np.array(nlls).mean(axis=0)
# y_pred = preds.mean.cpu().numpy()

In [22]:
print('Mean NLL:', nll)
print('MSE:', mean_squared_error(y_test, y_pred))
print('R2 score:',r2_score(y_test, y_pred))
mean, std = redshift_error(y_test, y_pred)
print(f'Redshift error: {mean:.5f} +/ {std:.5f}')

Mean NLL: -0.5980372697114944
MSE: 0.08720748035638085
R2 score: 0.838586897126171
Redshift error: 0.01240 +/ 0.12273
